# WasteLens - Iteration 12 (Colab GPU)


Compute env is Colab GPU; protocol unchanged from docs/rejection_experiment/iteration12_distillation_protocol.md (registered pre-training, commit 3b88e70). Run every code cell top to bottom; do not skip or reorder.

## S0 - environment

In [ ]:
!nvidia-smi


In [ ]:
import sys, platform
import tensorflow as tf
print('Python:', sys.version.split()[0], '|', platform.platform())
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus if gpus else 'NONE - enable a GPU runtime')
assert gpus, 'STOP: no GPU visible'
import numpy, sklearn, PIL
print('numpy:', numpy.__version__,
      '| sklearn:', sklearn.__version__,
      '| pillow:', PIL.__version__)


## S1 - repo sync (start from main, pin registered commit)

In [ ]:
import subprocess
from pathlib import Path
REPO = Path('/content/WasteLens')
PINNED = 'e017f34dbb487ec21a5633bd965543fbd8de7e9d'
if not (REPO / 'src' / 'train_distill_adaptation.py').exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/Anik00223/WasteLens.git',
                    str(REPO)], check=True)


In [ ]:
%cd /content/WasteLens
!git fetch origin main
!git checkout main
!git rev-parse HEAD
!git rev-parse origin/main
!git status --short


In [ ]:
import subprocess
head = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True,
                      text=True, cwd='/content/WasteLens').stdout.strip()
print('HEAD:', head)
assert head == 'e017f34dbb487ec21a5633bd965543fbd8de7e9d', 'STOP: HEAD != pinned commit'
print('repo pinned OK')


## S2 - dataset + teacher locks

In [ ]:
%cd /content/WasteLens
!ls scratch/adaptation_set | wc -l
!ls scratch/fresh_set | wc -l


In [ ]:
import sys
sys.path.insert(0, 'src')
import train_head_adaptation as th
print(th.lock_dataset())
import train_adaptation as wl10
print('shipped md5:', wl10.md5_file(th.SHIPPED_CHECKPOINT))


## S3 - train seed 42 (6 epochs, FINAL epoch = candidate)

In [ ]:
%cd /content/WasteLens
!python -u src/train_distill_adaptation.py --seed 42


## S4 - train seed 43 (identical recipe)

In [ ]:
%cd /content/WasteLens
!python -u src/train_distill_adaptation.py --seed 43


## S5 - evaluate both seeds (threshold 0.0702, gates per protocol)

In [ ]:
%cd /content/WasteLens
!python -u src/eval_head_adaptation.py --seed 42 --tag head12 --protocol iteration12_distillation_protocol.md --gates head12_gates.json
!python -u src/eval_head_adaptation.py --seed 43 --tag head12 --protocol iteration12_distillation_protocol.md --gates head12_gates.json


In [ ]:
%cd /content/WasteLens
!python -u src/eval_head_adaptation.py --summary --tag head12 --protocol iteration12_distillation_protocol.md


## S6 - evidence bundle (download to laptop)

In [ ]:
%cd /content/WasteLens
!git status --short
!ls -la docs/rejection_experiment/head12*
!tar -czf /content/head12_evidence.tar.gz docs/rejection_experiment/head12* docs/rejection_experiment/iteration12_decision.md
!ls -la /content/head12_evidence.tar.gz


In [ ]:
from google.colab import files
files.download('/content/head12_evidence.tar.gz')
